# 15.2 - LLM Evaluation

Status: VERIFIED

## What Are We Solving?
LLMs generate free-form text. Traditional metrics like accuracy don't apply. This unit covers reference-based metrics (BLEU, ROUGE), LLM-as-judge patterns, and multi-dimensional evaluation.

## Mental Model
LLM evaluation is multi-dimensional: correctness, helpfulness, safety, and cost are all different axes.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import json
from collections import Counter
print("All imports OK")

All imports OK


## Reference-Based Metrics: BLEU and ROUGE

In [2]:
# BLEU score (simplified implementation)
def compute_bleu(reference: str, hypothesis: str, max_n: int = 4) -> float:
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    if len(hyp_tokens) == 0:
        return 0.0
    
    scores = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter([tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1)])
        hyp_ngrams = Counter([tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens)-n+1)])
        
        matches = sum((hyp_ngrams & ref_ngrams).values())
        total = max(sum(hyp_ngrams.values()), 1)
        scores.append(matches / total)
    
    # Geometric mean
    log_avg = np.mean([np.log(max(s, 1e-10)) for s in scores])
    brevity_penalty = min(1.0, np.exp(1 - len(ref_tokens) / max(len(hyp_tokens), 1)))
    return brevity_penalty * np.exp(log_avg)

# Test
ref = "The capital of France is Paris and it is a beautiful city"
hyp1 = "Paris is the capital of France and it is beautiful"
hyp2 = "The capital of France is London"

print(f"Good hypothesis:  BLEU = {compute_bleu(ref, hyp1):.3f}")
print(f"Wrong hypothesis: BLEU = {compute_bleu(ref, hyp2):.3f}")

Good hypothesis:  BLEU = 0.340
Wrong hypothesis: BLEU = 0.280


In [3]:
# ROUGE score (simplified)
def compute_rouge_l(reference: str, hypothesis: str) -> float:
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    
    # Longest Common Subsequence
    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_len = dp[m][n]
    precision = lcs_len / max(n, 1)
    recall = lcs_len / max(m, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-10)
    return f1

ref = "The model uses attention mechanism to weight input features"
hyp = "The model applies attention to weight important features"
print(f"ROUGE-L: {compute_rouge_l(ref, hyp):.3f}")

ROUGE-L: 0.706


## LLM-as-Judge Pattern

In [4]:
# LLM-as-judge evaluation framework (mock implementation)
def mock_llm_judge(query: str, answer: str, criteria: list[str]) -> dict:
    """Simulate LLM-as-judge scoring. In production, call a real LLM API."""
    scores = {}
    for criterion in criteria:
        # Mock scoring based on simple heuristics
        if criterion == "relevance":
            query_words = set(query.lower().split())
            answer_words = set(answer.lower().split())
            overlap = len(query_words & answer_words) / max(len(query_words), 1)
            scores[criterion] = min(1.0, overlap + 0.3)
        elif criterion == "completeness":
            scores[criterion] = min(1.0, len(answer.split()) / 20)
        elif criterion == "safety":
            dangerous = ["hack", "exploit", "harmful", "illegal"]
            scores[criterion] = 1.0 if not any(d in answer.lower() for d in dangerous) else 0.2
        elif criterion == "clarity":
            sentences = answer.split('.')
            scores[criterion] = min(1.0, 0.5 + len(sentences) * 0.1)
    return {"scores": scores, "overall": np.mean(list(scores.values()))}

result = mock_llm_judge(
    "What is machine learning?",
    "Machine learning is a subset of AI where systems learn from data to make predictions.",
    ["relevance", "completeness", "safety", "clarity"]
)
print(f"Scores: {json.dumps(result['scores'], indent=2)}")
print(f"Overall: {result['overall']:.3f}")

Scores: {
  "relevance": 0.8,
  "completeness": 0.75,
  "safety": 1.0,
  "clarity": 0.7
}
Overall: 0.812


## Multi-Dimension Evaluation

In [5]:
# Multi-dimension evaluation framework
def evaluate_llm_output(query: str, answer: str, reference: str = None) -> dict:
    results = {}
    
    # 1. Reference-based (if reference provided)
    if reference:
        results["bleu"] = compute_bleu(reference, answer)
        results["rouge_l"] = compute_rouge_l(reference, answer)
    
    # 2. Quality dimensions
    judge = mock_llm_judge(query, answer, ["relevance", "completeness", "safety", "clarity"])
    results["judge_scores"] = judge["scores"]
    
    # 3. Hallucination check (simple: claims vs known facts)
    results["hallucination_risk"] = "low" if len(answer.split()) < 50 else "medium"
    
    # 4. Cost estimation
    results["token_estimate"] = len(answer.split()) * 4
    
    return results

eval_result = evaluate_llm_output(
    "Explain gradient descent",
    "Gradient descent is an optimization algorithm that iteratively adjusts parameters to minimize a loss function by moving in the direction of steepest descent.",
    "Gradient descent minimizes loss by updating parameters in the negative gradient direction."
)
print(json.dumps({k: v for k, v in eval_result.items() if k != "judge_scores"}, indent=2))
print(f"Judge scores: {eval_result['judge_scores']}")

{
  "bleu": 4.078442837439493e-06,
  "rouge_l": 0.3428571428571428,
  "hallucination_risk": "low",
  "token_estimate": 92
}
Judge scores: {'relevance': 0.9666666666666666, 'completeness': 1.0, 'safety': 1.0, 'clarity': 0.7}


In [6]:
# Verification
assert compute_bleu(ref, hyp1) > compute_bleu(ref, hyp2), "BLEU should rank correct answer higher"
assert result["overall"] > 0.3, "Judge score too low"
print("VERIFICATION PASSED: Phase 15.2 complete")

VERIFICATION PASSED: Phase 15.2 complete
